# Chains (LangChain v1.2)

**LCEL(LangChain Expression Language)**을 사용해서 모든 구성 요소가 `Runnable` 인터페이스로 통합되어 파이프라인(`|`)으로 연결될 수 있다.

```python
chain = prompt | model | output_parser  # 기본 구조
```

**구성 요소 업데이트 (v1.2 기준)**
1. **PromptTemplate**  
   - `Runnable`로 변환되어 LCEL 파이프라인에 직접 통합  
   ```python
   prompt = ChatPromptTemplate.from_template("...")
   ```

2. **LLM/ChatModel**  
   - `ChatOpenAI`, `ChatAnthropic` 등이 `Runnable` 구현  
   ```python
   model = ChatOpenAI(model="gpt-4o")
   ```

3. **Memory**  
   - `RunnableWithMessageHistory`로 통합 관리 (또는 LangGraph Persistence 사용)
   ```python
   chain_with_memory = RunnableWithMessageHistory(
       base_chain,
       get_session_history
   )
   ```

4. **Output Parsers**  
   - `StrOutputParser()`, `JsonOutputParser()` 등이 `Runnable`로 작동  
   ```python
   output_parser = JsonOutputParser()
   ```

5. **Tools**  
   - `@tool` 데코레이터로 생성 후 `RunnableLambda`로 변환  
   ```python
   @tool
   def search(query: str) -> str: ...
   ```

**체인 유형별 구현**


1. Simple Chain  

    ```python
    chain = prompt | model | output_parser
    response = chain.invoke({"input": "..."})
    ```

2. Sequential Chain  

    ```python
    chain = (
        {"step1_output": prompt1 | model1}  # 첫 번째 체인 결과 매핑
        | prompt2
        | model2
    )
    ```

3. Conditional Chain
    - `RunnableBranch` 사용

    ```python
    branch = RunnableBranch(
        (lambda x: x["topic"] == "math", math_chain),
        (lambda x: x["topic"] == "history", history_chain),
        default_chain
    )
    ```

4. Memory Chain  

    ```python
    memory_chain = RunnableWithMessageHistory(
        core_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history"
    )
    ```


**🚨 v1.2 주요 변경점**

- **Legacy Chain 클래스 완전 폐기**: `LLMChain`, `SequentialChain` 등은 `langchain-classic`으로 이동되거나 삭제됨 → `Runnable` (LCEL)로 통합
- **에이전트 통합**: `create_agent` (LangGraph 기반)가 표준

In [1]:
# %pip install -Uqqq langchain langchain-openai langchain-community

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

### Simple Chain

In [3]:
from langchain_core.prompts import PromptTemplate    # prompt chain 구성
from langchain.chat_models import init_chat_model    # 모델 chain 구성 래퍼
from langchain_core.output_parsers import StrOutputParser # 답변 문자형 변환

prompt = PromptTemplate.from_template('{city}의 특산물은 무엇입니까?')
llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

chain = prompt | llm | output_parser
print(chain.invoke('강원도'))
# 프롬프트 템플릿 변수가 2개 이상일 경우 dict형으로 전달
# print(chain.invoke(input={'city':'강원도',...})) 
print()
print(chain.invoke(input={'city':'강원도'})) 

강원도의 대표적인 특산물은 다음과 같습니다.

- **감자**: 평창, 강릉 등에서 많이 생산
- **옥수수**: 찰옥수수로 유명
- **메밀**: 평창·봉평 메밀과 메밀국수
- **한우**: 횡성 한우
- **황태**: 인제·평창 대관령 황태
- **오징어**: 속초·동해 지역
- **곤드레**: 정선 곤드레나물
- **잣**: 홍천 잣
- **송이버섯**: 양양 송이
- **초당두부**: 강릉 초당두부

이 밖에도 강원도는 산나물, 더덕, 도라지, 사과와 복숭아 등도 유명합니다.

강원도의 대표적인 특산물은 다음과 같습니다.

- **감자**: 평창·강릉·홍천 등에서 많이 생산됩니다.
- **옥수수**: 홍천 찰옥수수, 정선 옥수수 등이 유명합니다.
- **곤드레**: 정선 곤드레나물과 곤드레밥이 대표적입니다.
- **황태**: 인제 용대리 황태가 유명합니다.
- **한우**: 횡성한우가 대표적입니다.
- **오징어·명태**: 동해안 지역의 수산물입니다.
- **더덕·산나물**: 횡성, 평창, 정선 등 산간 지역에서 많이 납니다.
- **메밀**: 평창 봉평 메밀이 유명하며 메밀국수와 메밀전병으로 즐깁니다.
- **인삼**: 홍천과 철원 인삼도 잘 알려져 있습니다.


In [4]:
prompt1 = PromptTemplate.from_template('다음 내용을 한글로 번역하세요. {eng_text}')
prompt2 = PromptTemplate.from_template('다음 내용을 요약하세요. {kor_text}')
llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

# 번역 체인
chain1 = prompt1 | llm | output_parser
eng_text = """
One limitation of LLMs is their lack of contextual information (e.g., access to some specific documents or emails). You can combat this by giving LLMs access to the specific external data.
For this, you first need to load the external data with a document loader. LangChain provides a variety of loaders for different types of documents ranging from PDFs and emails to websites and YouTube videos.
"""
print(chain1.invoke(eng_text))

chain2 = prompt2 | llm | output_parser

kor_text ="""
LLM의 한 가지 한계는 특정 문서나 이메일과 같은 맥락 정보를 갖고 있지 않다는 점입니다. 이를 해결하려면 LLM이 특정 외부 데이터에 접근할 수 있도록 해야 합니다.
이를 위해서는 먼저 문서 로더를 사용해 외부 데이터를 불러와야 합니다. LangChain은 PDF, 이메일부터 웹사이트, 유튜브 영상에 이르기까지 다양한 유형의 문서를 위한 여러 종류의 로더를 제공합니다.
"""

print(chain2.invoke(kor_text)) 

LLM의 한 가지 한계는 맥락 정보가 부족하다는 점입니다. 예를 들어 특정 문서나 이메일에 접근할 수 없습니다. 이러한 한계를 극복하려면 LLM이 특정 외부 데이터에 접근할 수 있도록 해야 합니다.

이를 위해 먼저 문서 로더(document loader)를 사용하여 외부 데이터를 불러와야 합니다. LangChain은 PDF와 이메일부터 웹사이트 및 YouTube 동영상에 이르기까지 다양한 유형의 문서를 지원하는 여러 로더를 제공합니다.
LLM은 특정 문서나 이메일 같은 외부 맥락 정보에 직접 접근할 수 없다는 한계가 있습니다. 이를 보완하려면 외부 데이터를 LLM에 연결해야 하며, LangChain에서는 문서 로더를 통해 PDF, 이메일, 웹사이트, 유튜브 영상 등 다양한 데이터를 불러올 수 있습니다.


In [5]:
# Sequential Chain
chain = chain1 | chain2
print(chain.invoke({'eng_text': eng_text}))

LLM은 특정 문서나 이메일 등 외부 데이터에 접근하지 못하는 한계가 있습니다. 이를 보완하려면 문서 로더를 통해 외부 데이터를 불러와야 하며, LangChain은 PDF, 이메일, 웹사이트, YouTube 동영상 등 다양한 형식의 로더를 제공합니다.


### Conditional Chain

In [6]:
from langchain_core.runnables import RunnableBranch

llm = init_chat_model('gpt-5.6-luna')

math_prompt = PromptTemplate.from_template('다음 문제를 풀어주세요. 단계적인 풀이를 수식(LaTex)과 함께 작성해주세요.{question}')
math_chain = math_prompt | llm | output_parser
default_prompt = PromptTemplate.from_template('당신은 친절하고, 감성적이며 공감능력이 좋은 챗봇입니다. 다음 질문에 답변해주세요. {question}')
default_chain = default_prompt | llm | output_parser

# math_chain 선택 함수
def is_math_question(input_dict: dict) -> bool:
    question: str = input_dict.get('question', '') # 입력받은 dict에서 question 키의 값을 추출(없으면 빈 문자열)
    return '계산' in question or 'calc' in question

branch_chain = RunnableBranch(
    (is_math_question,math_chain),
    default_chain
)

print(branch_chain.invoke({'question': '125 * 3 + 50 계산해줘.'}))

계산식:

\[
125 \times 3 + 50
\]

먼저 곱셈을 계산합니다.

\[
125 \times 3 = 375
\]

그다음 \(50\)을 더하면,

\[
375 + 50 = 425
\]

따라서 답은

\[
\boxed{425}
\]


In [7]:
print(branch_chain.invoke({'question': '나 오늘 우울해. 빵? 밥?'}))

오늘은 **따뜻한 밥** 어때? 🍚  
우울한 날엔 따뜻하고 든든한 게 마음까지 조금 감싸주는 느낌이 들 때가 있어. 계란 하나나 김만 있어도 충분해.

그래도 빵이 더 당기면 빵 먹어도 괜찮아. 오늘은 잘 먹는 것 자체가 잘한 일이니까. 💛


### Memory Chain

`RunnableWithMessageHistory`를 사용하여 대화내역을 기억하는 chain을 생성한다.

In [8]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from pydantic import BaseModel, Field
from typing import List

# 사용자별 세션 대화내역을 기록하는 클래스
class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    messages: List[BaseMessage] = Field(default_factory=list) # 인스턴스마다 독립적인 messages list를 구성

    def add_messages(self, messages: list[BaseMessage]) -> None:
        self.messages.extend(messages)
    def clear(self) -> None:
        self.messages = []

store = {} # {session_id: 히스토리 객체(InMemoryHistory)} 저장소

# 세션 ID로 히스토리 객체를 반환
def get_by_session_id(sesstion_id: str) -> BaseChatMessageHistory:
    if sesstion_id not in store:
        store[sesstion_id] = InMemoryHistory() # 기존 대화내역이 없으면 히스토리 객체 생성해서 store에 추가
    return store[sesstion_id] # 해당 세션의 히스토리 객체 반환

history1 = get_by_session_id('1')
history1.add_messages([AIMessage(content='반갑습니다. sungmin님!')])
history1.add_messages([HumanMessage(content='그래 나 min이야 반갑다!')])
print(f"{history1 = }")

history2 = get_by_session_id('2')
print(f"{history2 = }")

history1 = InMemoryHistory(messages=[AIMessage(content='반갑습니다. sungmin님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='그래 나 min이야 반갑다!', additional_kwargs={}, response_metadata={})])
history2 = InMemoryHistory(messages=[])


### 대화 히스토리를 자동으로 누적하는 Memory Chain (RunnableWithMessageHistory)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # 채팅 프롬프트 템플릿 / 히스토리 자리표시자
from langchain_core.runnables import RunnableWithMessageHistory # 실행시 히스토리를 붙여주는 Runnable

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {domain} 분야의 전문가 챗봇입니다.'),
    MessagesPlaceholder(variable_name='history'), # 세션별 이전 대화 메시지들이 들어갈 자리
    ('human', '{question}')
])

llm = init_chat_model('gpt-5.6-luna')

chain = prompt | llm

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_by_session_id,  # sessionID로 히스토리 객체 가져오는 함수 참조
    input_messages_key='question', # 입력 dict에서 question키의 값은 사용자 메시지
    history_messages_key='history' # 프롬프트에서 history 받을 변수명
)
chain_with_history.invoke({
    'domain': 'math',
    'question': '민수는 강아지를 3마리 키우고 있습니다.'
}, config={
    'configurable': {
        'session_id': '100'
    }
})


c:\Workspace\LLM\llm_venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


AIMessage(content='그렇군요! 민수는 강아지 3마리를 키우고 있네요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 38, 'total_tokens': 101, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 33, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHHxMOTt0JD6ww4BzIQemKRqS48Du', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0408b-f108-76b2-87b0-318eda822f69-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 63, 'total_tokens': 101, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 33}})

In [10]:
chain_with_history.invoke({
    'domain': 'math',
    'question': '소라는 고양이 4마리 키우고 있습니다.'
}, config={
    'configurable': {
        'session_id': '100'
    }
})

AIMessage(content='그렇군요! 소라는 고양이 4마리를 키우고 있네요. 민수와 소라가 키우는 반려동물은 모두 7마리입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 83, 'total_tokens': 177, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 44, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHI5Yst7Jqn6LabxFUZeLArBWZnHN', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a04093-acdc-75a1-b6af-639730fb4b0b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 83, 'output_tokens': 94, 'total_tokens': 177, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, '

In [12]:
store # 현재 메모리에 저장된 세션별 대화 히스토리

{'1': InMemoryHistory(messages=[AIMessage(content='반갑습니다. sungmin님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='그래 나 min이야 반갑다!', additional_kwargs={}, response_metadata={})]),
 '2': InMemoryHistory(messages=[]),
 '100': InMemoryHistory(messages=[HumanMessage(content='민수는 강아지를 3마리 키우고 있습니다.', additional_kwargs={}, response_metadata={}), AIMessage(content='그렇군요! 민수는 강아지 3마리를 키우고 있네요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 38, 'total_tokens': 101, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 33, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EHHxMOTt0JD6

### ChatMessageHistroy
- 대화 내용만 저장

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory

store = {}
# 세션 ID로 히스토리 객체를 반환
def get_by_session_id(sesstion_id: str) -> BaseChatMessageHistory:
    if sesstion_id not in store:
        store[sesstion_id] = ChatMessageHistory() # 기존 대화내역이 없으면 히스토리 객체 생성해서 store에 추가
    return store[sesstion_id] # 해당 세션의 히스토리 객체 반환

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {domain} 분야의 전문가 챗봇입니다.'),
    MessagesPlaceholder(variable_name='history'), # 세션별 이전 대화 메시지들이 들어갈 자리
    ('human', '{question}')
])

llm = init_chat_model('gpt-5.6-luna')

chain = prompt | llm | output_parser

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_by_session_id,  # sessionID로 히스토리 객체 가져오는 함수 참조
    input_messages_key='question', # 입력 dict에서 question키의 값은 사용자 메시지
    history_messages_key='history' # 프롬프트에서 history 받을 변수명
)
chain_with_history.invoke({
    'domain': '심리상담',
    'question': '요즘 갑자기 더워져서 짜증나는데? 나 성격 좋은데? 왜이러지?'
}, config={ # RunnableWithMessageHistory 설정
    'configurable': {
        'session_id': '200'
    }
})

c:\Workspace\LLM\llm_venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


'갑자기 더워지면 짜증이 늘 수 있어요. **성격이 나빠져서가 아니라**, 몸이 열을 식히느라 에너지를 쓰고, 수면의 질·집중력·수분 균형이 떨어지면서 감정 조절 여유가 줄어들기 때문입니다. 더위는 불편감과 피로를 높여 작은 일에도 예민하게 반응하게 만들 수 있어요.\n\n도움 되는 방법은:\n\n- 물을 자주 마시고, 땀을 많이 흘렸다면 전해질도 보충하기  \n- 실내 온도·습도 조절, 가볍고 통풍되는 옷 입기  \n- 더운 시간대에는 중요한 일이나 대화를 조금 미루기  \n- 짜증이 올라올 때 “지금 내가 화난 게 아니라 더위와 피로가 큰가?” 하고 잠깐 쉬기  \n- 수면 부족, 공복, 카페인 과다도 함께 점검하기\n\n다만 짜증이 **몇 주 이상 지속되거나**, 잠·식욕 변화, 심한 불안이나 우울, 두근거림·어지럼·무기력까지 동반되면 더위 외 다른 원인이 있을 수 있으니 상담이나 진료를 고려해 보세요.'

In [14]:
chain_with_history.invoke({
    'domain': '심리상담',
    'question': '그럼 니가 날씨를 좋게 만들어주면 되잖아'
}, config={ # RunnableWithMessageHistory 설정
    'configurable': {
        'session_id': '200'
    }
})

'그러게요, 제가 날씨까지 조절할 수 있으면 바로 선선하게 만들어드릴 텐데요 😅  \n대신 **지금 당장 덜 덥게 느끼는 방법**은 같이 찾아볼 수 있어요.\n\n- 찬물보다 **미지근한 물을 자주** 마시기  \n- 목·손목·겨드랑이처럼 혈관이 많은 부위를 시원하게 하기  \n- 선풍기는 창문 쪽으로 틀어 더운 공기를 먼저 빼기  \n- 에어컨은 너무 낮추기보다 **26~28도 정도**로 설정하기  \n- 짜증 날 때는 중요한 대화나 결정은 잠시 미루기\n\n날씨는 못 바꿔도, 더위가 당신 성격을 대신 결정하게 두지는 않을 수 있어요.'

In [17]:
print(store['200'])

Human: 요즘 가밪기 더워져서 짜증나는데? 나 성격 좋은데? 왜이러지?
AI: 갑자기 더워지면 짜증이 늘 수 있어요. **성격이 나빠져서가 아니라**, 몸이 열을 식히느라 에너지를 쓰고, 수면의 질·집중력·수분 균형이 떨어지면서 감정 조절 여유가 줄어들기 때문입니다. 더위는 불편감과 피로를 높여 작은 일에도 예민하게 반응하게 만들 수 있어요.

도움 되는 방법은:

- 물을 자주 마시고, 땀을 많이 흘렸다면 전해질도 보충하기  
- 실내 온도·습도 조절, 가볍고 통풍되는 옷 입기  
- 더운 시간대에는 중요한 일이나 대화를 조금 미루기  
- 짜증이 올라올 때 “지금 내가 화난 게 아니라 더위와 피로가 큰가?” 하고 잠깐 쉬기  
- 수면 부족, 공복, 카페인 과다도 함께 점검하기

다만 짜증이 **몇 주 이상 지속되거나**, 잠·식욕 변화, 심한 불안이나 우울, 두근거림·어지럼·무기력까지 동반되면 더위 외 다른 원인이 있을 수 있으니 상담이나 진료를 고려해 보세요.
Human: 그럼 니가 날씨를 좋게 만들어주면 되잖아
AI: 그러게요, 제가 날씨까지 조절할 수 있으면 바로 선선하게 만들어드릴 텐데요 😅  
대신 **지금 당장 덜 덥게 느끼는 방법**은 같이 찾아볼 수 있어요.

- 찬물보다 **미지근한 물을 자주** 마시기  
- 목·손목·겨드랑이처럼 혈관이 많은 부위를 시원하게 하기  
- 선풍기는 창문 쪽으로 틀어 더운 공기를 먼저 빼기  
- 에어컨은 너무 낮추기보다 **26~28도 정도**로 설정하기  
- 짜증 날 때는 중요한 대화나 결정은 잠시 미루기

날씨는 못 바꿔도, 더위가 당신 성격을 대신 결정하게 두지는 않을 수 있어요.


##### 세션(메모리) 방식의 문제점
- 메모리 저장이라 영속성이 없음
    - 서버 재시작/재배포 하면 store가 날아가서 히스토리도 같이 사라짐
- 세션 식별이 끊기기 쉬움
    - 쿠키/세션ID가 유지되지 않으면 같은 사람인지 매칭이 안 됨
- 스케일 아웃(서버 여러 대)에서 깨짐
    - A서버 메모리에 저장된 히스토리를 B서버는 모름 → 대화가 끊김

그래서 보통 이렇게 구성한다.
- SQLite/Redis/RDB 같은 저장소에 대화 내역을 저장해서
    - 사용자가 재접속해도 user_id 또는 thread_id로 복원
- 프롬프트에는 보통
    - 최근 N턴 + 요약 형태로 넣어서 비용/토큰도 관리